# Two-time and singletime response at a probe time
Compute a Gaussian double-time response and a one-sided relative-time response using separate lesser/greater CSVs. Run from this notebook's directory. The two methods have different windows, prefactors, and normalizations: compare their shapes with care. Neither includes a core-hole XAS Hamiltonian, a spatial transform, or a gauge correction.

The two-time input must contain complete rectangular `t, t_prime` grids. The singletime input may be a combined file or the per-step file for the chosen probe time. Inputs are raw correlators without an i prefactor.

In [ ]:
from pathlib import Path
import csv
import sys
import numpy as np
import matplotlib.pyplot as plt

code_dir = Path.cwd() / 'code'  # Or set to the full path of this folder's code/
if not (code_dir / 'plotting_td_dos.py').is_file():
    raise FileNotFoundError(f'Code directory not found: {code_dir.resolve()}')
sys.path.insert(0, str(code_dir.resolve()))
from plotting_td_dos import fixed_probe_response

mode = 'both'  # 'lesser', 'greater', or 'both'
two_lesser = Path('path/to/G<_twotime.csv')
two_greater = Path('path/to/G>_twotime.csv')
single_lesser = Path('path/to/G<_singletime.csv')  # Combined or matching step CSV
single_greater = Path('path/to/G>_singletime.csv')
site_pair = (1, 1)
probe_time = 8.0  # Must be an existing t_prime in the singletime input
sigma = 3.0  # Double-time Gaussian probe width
relative_sigma = sigma * np.sqrt(2)  # One-sided Gaussian relative-time width
omega_min, omega_max, omega_count = -15.0, 15.0, 401
frequency_shift_lesser = -4.0  # Displayed frequency = evaluated frequency - shift
frequency_shift_greater = -4.0
output_csv = Path('td_dos_probe_response.csv')


## Calculate at the selected probe time
The singletime value of `t_prime` must match `probe_time` within 1e−9; no nearest-time substitution or interpolation is performed. Both methods use matching Fourier signs: lesser uses +iω(t−t′), and greater uses −iω(t−t′). Components are summed only after their separate transforms. The CSV output is overwritten.

In [ ]:
if mode not in ('lesser', 'greater', 'both'):
    raise ValueError("mode must be 'lesser', 'greater', or 'both'")
kinds = ('lesser', 'greater') if mode == 'both' else (mode,)
two_all = {'lesser': two_lesser, 'greater': two_greater}
single_all = {'lesser': single_lesser, 'greater': single_greater}
two_inputs = {k: two_all[k] for k in kinds}
single_inputs = {k: single_all[k] for k in kinds}
for path in [*two_inputs.values(), *single_inputs.values()]:
    if not path.is_file():
        raise FileNotFoundError(f'Input not found: {path.resolve()}')
    if output_csv.resolve() == path.resolve():
        raise ValueError('Output CSV must not overwrite an input')
if omega_count < 2 or omega_min >= omega_max:
    raise ValueError('Use at least two frequencies and omega_min < omega_max')
frequencies = np.linspace(omega_min, omega_max, omega_count)
shifts = {'lesser': frequency_shift_lesser, 'greater': frequency_shift_greater}
result = fixed_probe_response(two_inputs, single_inputs, site_pair, probe_time, frequencies, sigma, relative_sigma, shifts)
low, high = result['nyquist_display_bounds']
display_min, display_max = max(omega_min, low), min(omega_max, high)
if display_min >= display_max:
    raise ValueError('Requested frequency grid does not intersect the common Nyquist window')
print(f'Common shifted Nyquist display window: [{low:g}, {high:g}]')
print(f'Displayed frequency window: [{display_min:g}, {display_max:g}]')
with output_csv.open('w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(('center_site', 'site_index', 'probe_time', 'omega_shifted', 'two_time_real', 'two_time_imag', 'relative_response_real', 'relative_response_imag', 'relative_intensity'))
    for w, two, relative, intensity in zip(frequencies, result['two_time'], result['relative_response'], result['relative_intensity']):
        writer.writerow((*site_pair, probe_time, w, two.real, two.imag, relative.real, relative.imag, intensity))
print(f'Wrote {output_csv}')


## Compare frequency responses
The left plot shows the raw complex double-time transform. The right plot shows the one-sided intensity, −Im(response)/π. Each frequency axis uses the common shifted Nyquist window; plotting inside this window does not remove aliasing already in the input.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True, constrained_layout=True)
axes[0].plot(frequencies, result['two_time'].real, label='Real')
axes[0].plot(frequencies, result['two_time'].imag, label='Imaginary')
axes[0].set(title=f'Two-time ({mode})', ylabel='Gaussian transform')
axes[1].plot(frequencies, result['relative_intensity'], label='Intensity')
axes[1].set(title=f'Singletime ({mode})', ylabel='−Im(response) / π')
for ax in axes:
    ax.set(xlabel='Shifted frequency', xlim=(display_min, display_max))
    ax.legend()
    ax.grid(alpha=0.3)
fig.suptitle(f'Sites {site_pair}; probe time {probe_time:g}')
plt.show()


## Inspect singletime input slices
This plots the raw forward correlators before the relative-time transform.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True, constrained_layout=True)
for kind, data in result['time_slices'].items():
    axes[0].plot(data['tau'], data['correlator'].real, label=kind)
    axes[1].plot(data['tau'], data['correlator'].imag, label=kind)
for ax, label in zip(axes, ('Real part', 'Imaginary part')):
    ax.set(xlabel='Relative time τ', ylabel=label)
    ax.legend()
    ax.grid(alpha=0.3)
fig.suptitle(f'Raw singletime correlators, t′ = {probe_time:g}, sites {site_pair}')
plt.show()
